# Bibliotheque + lecture Excel

In [1]:
import pandas as pd 
import numpy as np 
import os
import re
import pandas as pd
from collections import defaultdict
import zipfile
from datetime import datetime, timedelta
from pathlib import Path
import datetime

In [2]:
df = pd.read_excel('AGING_Ceinture_Montre (avec groupe).xlsx',sheet_name=None)

C:\Users\judupont\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():
C:\Users\judupont\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [3]:
df.keys()

dict_keys(['APHM', 'CAEN', 'POITIERS', 'ROUEN', 'LAVERAN', 'LPC', 'SAINTE-MARGUERITE', 'CGD', 'CERCA', 'Effectifs ceinture', 'Effectifs montre', 'CODEBOOK '])

# Vérification blocs 

In [4]:
def convertir_heure_excel_ou_texte(x):
    """
    Convertit une valeur Excel ou texte en datetime, en ne gardant que l'heure.
    """
    if pd.isna(x):
        return pd.NaT

    # Cas nombre Excel (jours depuis 1899-12-30)
    if isinstance(x, (int, float)):
        ts = pd.to_datetime(x, unit="D", origin="1899-12-30")
        return ts.replace(year=1900, month=1, day=1)  # ignore la vraie date

    x_str = str(x).strip()

    # Cas hh:mm ou hh:mm:ss
    for fmt in ("%H:%M:%S", "%H:%M"):
        try:
            ts = pd.to_datetime(x_str, format=fmt)
            return ts.replace(year=1900, month=1, day=1)  # force date fixe
        except Exception:
            continue

    # Cas général (Excel parfois avec date complète)
    try:
        ts = pd.to_datetime(x_str, errors="coerce")
        if pd.isna(ts):
            return pd.NaT
        return ts.replace(year=1900, month=1, day=1)  # ignore la date réelle
    except Exception:
        return pd.NaT

In [7]:
def analyser_feuille_bloc(
    df,
    nom_feuille,
    col_debut,
    col_fin,
    nom_bloc
):
    print(f"\n--- Feuille : {nom_feuille} | {nom_bloc} ---")

    col_id = "Numero_inclusion"

    # ===============================
    # 🔧 DEBUG : normalisation ID
    # ===============================
    df = df.copy()
    df[col_id] = df[col_id].astype(str).str.upper().str.strip()

    # ==================================================
    # Dictionnaire des colonnes atitrées manuellement 
    # ==================================================
    REGLES_CANDIDATS = {
        ("CAEN", "bloc3", "0303PAR"): {"col_debut": "heure_rappel_cond_v2"},

        ("CAEN", "bloc3", "0305LCR"): {"col_debut": "heure_fluence_sem_v2"},

        ("CAEN", "bloc2S", "0314CDS"): {"col_fin": "heure_fluence_sem_v2"},
        ("CAEN", "bloc3", "0314CDS"): {"col_debut": "heure_fluence_sem_v2"},
        
        ("CAEN", "bloc2S", "0335JMS"): {"col_fin": "heure_fluence_sem_v2"},
        ("CAEN", "bloc3", "0335JMS"): {"col_debut": "heure_fluence_sem_v2"},

        ("CAEN", "bloc3", "0339NPR"): {"col_debut": "heure_fluence_sem_v2"},

        ("CAEN", "bloc2S", "0346GLS"): { "col_fin": "heure_fluence_sem_v2"},
        ("CAEN", "bloc3", "0346GLS"): {"col_debut": "heure_fluence_sem_v2"},

        ("CAEN", "bloc2R", "0353LNR"): { "col_fin": "heure_fluence_sem_v2"},
        ("CAEN", "bloc3", "0353LNR"): {"col_debut": "heure_fluence_sem_v2"},

        ("POITIERS", "bloc2S", "0414PJS"): {"col_fin": "heure_fluence_sem_v2"},
        ("POITIERS", "bloc3", "0414PJS"): {"col_debut": "heure_fluence_sem_v2"},

        ("CGD", "bloc2S", "0902SIS"): {"col_fin": "heure_fluence_sem_v2"},
        ("CGD", "bloc3", "0902SIS"): {"col_debut": "heure_fluence_sem_v2"},

        ("CGD", "bloc3", "0905SLR"): {"col_debut": "heure_nback_v2"},

        ("CGD", "bloc3", "0906DNR"): {"col_debut": "heure_fluence_sem_v2"},

        ("CERCA", "bloc3", "0414PVR"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc3", "0419SRR"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc3", "0420MCR"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc2S", "0421ACS"): {"col_fin": "heure_nback_v2"},
        ("CERCA", "bloc3", "0421ACS"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc3", "0423RMR"): {"col_debut": "heure_nback_v2"},

        ("CERCA", "bloc3", "0424BLR"): {"col_debut": "heure_nback_v2"},
    }

    # ==========================================================
    # Colonnes effecives
    # ==========================================================
    df["_col_debut"] = col_debut
    df["_col_fin"] = col_fin

    # ==========================================================
    # RÈGLES GLOBALES
    # ==========================================================
    if nom_feuille == "SAINTE-MARGUERITE" and nom_bloc == "bloc2S":
        df["_col_fin"] = "heure_nback_v2"

    if nom_feuille == "SAINTE-MARGUERITE" and nom_bloc == "bloc3":
        df["_col_debut"] = "heure_nback_v2"

    # ==========================================================
    # RÈGLES SPÉCIFIQUES (écrasent le global)
    # ==========================================================
    print("\n🔎 Vérification règles spécifiques :")

    for (centre, bloc, candidat), regle in REGLES_CANDIDATS.items():

        if centre != nom_feuille or bloc != nom_bloc:
            continue

        mask = df[col_id] == candidat

        if mask.any():
            print(f"🔧 Règle appliquée pour {candidat}")
            if "col_debut" in regle:
                df.loc[mask, "_col_debut"] = regle["col_debut"]
            if "col_fin" in regle:
                df.loc[mask, "_col_fin"] = regle["col_fin"]
        else:
            print(f"⚠️ {candidat} non trouvé dans cette feuille")

    # ==========================================================
    # DEBUG COLONNES UTILISÉES
    # ==========================================================
    print("\n🔎 Colonnes finales utilisées (extrait) :")
    print(df[[col_id, "_col_debut", "_col_fin"]].head())

    # ==========================================================
    # 6️⃣ CONVERSION HEURES
    # ==========================================================
    df["_debut_ts"] = df.apply(
        lambda r: convertir_heure_excel_ou_texte(r[r["_col_debut"]]),
        axis=1
    )

    df["_fin_ts"] = df.apply(
        lambda r: convertir_heure_excel_ou_texte(r[r["_col_fin"]]),
        axis=1
    )
    # ===== DEBUG candidat spécifique =====
    print(
        df.loc[df["Numero_inclusion"] == "0101EMS",
           ["Numero_inclusion", "_col_debut", "_col_fin", "_debut_ts", "_fin_ts"]]
    )
    # ==========================================================
    # 7️⃣ CALCUL DURÉE
    # ==========================================================
    col_duree = f"duree_{nom_bloc}"
    col_rejet = f"rejeter_{nom_bloc}"

    df[col_duree] = (df["_fin_ts"] - df["_debut_ts"]).dt.total_seconds() / 60
    df.loc[df[col_duree] < 0, col_duree] += 24 * 60

    # ==========================================================
    # 8️⃣ STATISTIQUES
    # ==========================================================
    mask_manquant = df[col_duree].isna()
    mask_valide = df[col_duree].notna() & (df[col_duree] > 0)

    moyenne = df.loc[mask_valide, col_duree].mean()
    ecart_type = df.loc[mask_valide, col_duree].std()

    print(f"\nMoyenne = {moyenne:.2f} min")
    print(f"Écart-type = {ecart_type:.2f} min")

    borne_inf = moyenne - 2 * ecart_type

    df[col_rejet] = 0
    df.loc[mask_manquant, col_rejet] = 2

    mask_rejet = df[col_duree] < borne_inf

    if nom_bloc == "bloc1":
        mask_rejet = mask_rejet | (df[col_duree] < 10)

    df.loc[mask_valide & mask_rejet, col_rejet] = 1

    # ==========================================================
    # 9️⃣ DEBUG CANDIDATS PROBLÉMATIQUES
    # ==========================================================
    print("\n🔎 DEBUG Candidats spécifiques :")

    candidats_test = [r[2] for r in REGLES_CANDIDATS.keys()]
    candidats_test = list(set(candidats_test))

    print(
        df.loc[
            df[col_id].isin(candidats_test),
            [col_id, col_duree, col_rejet]
        ]
    )

    # ==========================================================
    # 10️⃣ NETTOYAGE
    # ==========================================================
    df = df.drop(columns=["_col_debut", "_col_fin", "_debut_ts", "_fin_ts"])

    print(f"\nRésumé {col_rejet} :")
    print(df[col_rejet].value_counts().sort_index())

    return {
        "df": df,
        "moyenne": moyenne,
        "ecart_type": ecart_type,
        "rejets": {
            "manquant": df.loc[df[col_rejet] == 2, col_id].tolist(),
            "rejetes": df.loc[df[col_rejet] == 1, col_id].tolist()
        }
    }

## Bloc 1

In [8]:
feuilles = [
    'APHM', 'CAEN', 'POITIERS', 'ROUEN',
    'LAVERAN', 'LPC', 'SAINTE-MARGUERITE',
    'CGD', 'CERCA'
]

resultats_bloc1 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc1[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_ceinture_v2",   
        col_fin="heure_fin_anamnese_v2",        
        nom_bloc="bloc1"
    )




--- Feuille : APHM | bloc1 ---

🔎 Vérification règles spécifiques :

🔎 Colonnes finales utilisées (extrait) :
  Numero_inclusion         _col_debut               _col_fin
0          0101CAR  heure_ceinture_v2  heure_fin_anamnese_v2
1          0102PCR  heure_ceinture_v2  heure_fin_anamnese_v2
2          0103SHS  heure_ceinture_v2  heure_fin_anamnese_v2
3          0105PNR  heure_ceinture_v2  heure_fin_anamnese_v2
4          0104FJS  heure_ceinture_v2  heure_fin_anamnese_v2
Empty DataFrame
Columns: [Numero_inclusion, _col_debut, _col_fin, _debut_ts, _fin_ts]
Index: []

Moyenne = 25.36 min
Écart-type = 10.48 min

🔎 DEBUG Candidats spécifiques :
Empty DataFrame
Columns: [Numero_inclusion, duree_bloc1, rejeter_bloc1]
Index: []

Résumé rejeter_bloc1 :
rejeter_bloc1
0    25
Name: count, dtype: int64


--- Feuille : CAEN | bloc1 ---

🔎 Vérification règles spécifiques :

🔎 Colonnes finales utilisées (extrait) :
  Numero_inclusion         _col_debut               _col_fin
0          0301GNR  he

## Bloc 2 

### Avec vidéo: de heure_rlri16imm_debut_V2 à heure_rappel_cond_v2

In [7]:
resultats_bloc2_R = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc2_R[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_rlri16imm_debut_V2",  
        col_fin= "heure_rappel_cond_v2" ,  # Beaucoup de rejet car tous les S ont des NV car pas vu la vidéo         
        nom_bloc="bloc2R"
    )




--- Feuille : APHM | bloc2R ---

🔎 Vérification règles spécifiques :

🔎 Colonnes finales utilisées (extrait) :
  Numero_inclusion                _col_debut              _col_fin
0          0101CAR  heure_rlri16imm_debut_V2  heure_rappel_cond_v2
1          0102PCR  heure_rlri16imm_debut_V2  heure_rappel_cond_v2
2          0103SHS  heure_rlri16imm_debut_V2  heure_rappel_cond_v2
3          0105PNR  heure_rlri16imm_debut_V2  heure_rappel_cond_v2
4          0104FJS  heure_rlri16imm_debut_V2  heure_rappel_cond_v2
Empty DataFrame
Columns: [Numero_inclusion, _col_debut, _col_fin, _debut_ts, _fin_ts]
Index: []

Moyenne = 44.92 min
Écart-type = 7.48 min

🔎 DEBUG Candidats spécifiques :
Empty DataFrame
Columns: [Numero_inclusion, duree_bloc2R, rejeter_bloc2R]
Index: []

Résumé rejeter_bloc2R :
rejeter_bloc2R
0    13
2    12
Name: count, dtype: int64


--- Feuille : CAEN | bloc2R ---

🔎 Vérification règles spécifiques :
🔧 Règle appliquée pour 0353LNR

🔎 Colonnes finales utilisées (extrait) :
  N

resultats_bloc2_R

### Sans vidéo: de heure_rlri16imm_debut_V2 à heure_nback_v2

In [8]:
resultats_bloc2_S = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc2_S[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_rlri16imm_debut_V2",
        col_fin="heure_nback_v2 (consigne)",
        nom_bloc="bloc2S"
    )



--- Feuille : APHM | bloc2S ---

🔎 Vérification règles spécifiques :

🔎 Colonnes finales utilisées (extrait) :
  Numero_inclusion                _col_debut                   _col_fin
0          0101CAR  heure_rlri16imm_debut_V2  heure_nback_v2 (consigne)
1          0102PCR  heure_rlri16imm_debut_V2  heure_nback_v2 (consigne)
2          0103SHS  heure_rlri16imm_debut_V2  heure_nback_v2 (consigne)
3          0105PNR  heure_rlri16imm_debut_V2  heure_nback_v2 (consigne)
4          0104FJS  heure_rlri16imm_debut_V2  heure_nback_v2 (consigne)
Empty DataFrame
Columns: [Numero_inclusion, _col_debut, _col_fin, _debut_ts, _fin_ts]
Index: []

Moyenne = 46.44 min
Écart-type = 9.26 min

🔎 DEBUG Candidats spécifiques :
Empty DataFrame
Columns: [Numero_inclusion, duree_bloc2S, rejeter_bloc2S]
Index: []

Résumé rejeter_bloc2S :
rejeter_bloc2S
0    25
Name: count, dtype: int64


--- Feuille : CAEN | bloc2S ---

🔎 Vérification règles spécifiques :
🔧 Règle appliquée pour 0314CDS
🔧 Règle appliquée pour 

## Bloc 3 

In [9]:
resultats_bloc3 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc3[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_nback_v2 (consigne)",   
        col_fin="heure_fin_tests_v2",        
        nom_bloc="bloc3"
    )



--- Feuille : APHM | bloc3 ---

🔎 Vérification règles spécifiques :

🔎 Colonnes finales utilisées (extrait) :
  Numero_inclusion                 _col_debut            _col_fin
0          0101CAR  heure_nback_v2 (consigne)  heure_fin_tests_v2
1          0102PCR  heure_nback_v2 (consigne)  heure_fin_tests_v2
2          0103SHS  heure_nback_v2 (consigne)  heure_fin_tests_v2
3          0105PNR  heure_nback_v2 (consigne)  heure_fin_tests_v2
4          0104FJS  heure_nback_v2 (consigne)  heure_fin_tests_v2
Empty DataFrame
Columns: [Numero_inclusion, _col_debut, _col_fin, _debut_ts, _fin_ts]
Index: []

Moyenne = 48.84 min
Écart-type = 8.85 min

🔎 DEBUG Candidats spécifiques :
Empty DataFrame
Columns: [Numero_inclusion, duree_bloc3, rejeter_bloc3]
Index: []

Résumé rejeter_bloc3 :
rejeter_bloc3
0    24
1     1
Name: count, dtype: int64


--- Feuille : CAEN | bloc3 ---

🔎 Vérification règles spécifiques :
🔧 Règle appliquée pour 0303PAR
🔧 Règle appliquée pour 0305LCR
🔧 Règle appliquée pour 031

# Nombre de candidats qui passent toutes les conditions 

In [10]:
def candidats_valides_tous_blocs_depuis_resultats(
    feuille,
    resultats_blocs,
    col_id="Numero_inclusion"
):
    """
    Un candidat est conservé si nous pouvons calculer la durée de tous ses blocs et si celles-ci respectent les conditions suivantes :
    - bloc 1 > 10 min
    - bloc2R/S et bloc 3 > mean - 2σ min
    
    resultats_blocs = dict {
        "bloc1": resultats_bloc1,
        "bloc2S": resultats_bloc2_S,
        "bloc2R": resultats_bloc2_R,
        "bloc3": resultats_bloc3
    }
    """

    # Récupération du df de référence (bloc1 par ex)
    df_ref = resultats_blocs["bloc1"][feuille]["df"].copy()

    colonnes_rejet = []

    # Ajouter chaque colonne de rejet depuis chaque bloc
    for nom_bloc, res_bloc in resultats_blocs.items():
        col_rejet = f"rejeter_{nom_bloc}"
        if col_rejet not in res_bloc[feuille]["df"].columns:
            print(f"Colonne manquante : {col_rejet} dans {feuille}")
            return None

        df_ref[col_rejet] = res_bloc[feuille]["df"][col_rejet]
        colonnes_rejet.append(col_rejet)

    # Condition : tout à 0
    mask_valide = (df_ref[colonnes_rejet] == 0).all(axis=1)

    candidats_ok = df_ref.loc[mask_valide, col_id].tolist()
    candidats_rejetes = df_ref.loc[~mask_valide, col_id].tolist()

    print(f"\n=== {feuille} ===")
    print(f"Candidats valides sur TOUS les blocs : {len(candidats_ok)}")

    print("Liste des candidats conservés :")
    for pid in candidats_ok:
        print(f" - {pid}")

    return {
        "nb_valides": len(candidats_ok),
        "valides": candidats_ok,
        "rejetes": candidats_rejetes,
    }


In [11]:
resultats_globaux = {}

colonnes_blocs = [
    "rejeter_bloc1",
    "rejeter_bloc2S",
    "rejeter_bloc2R",
    "rejeter_bloc3"
]

for feuille in feuilles:
    print("\n" + "=" * 70)
    print(f"Analyse globale – Feuille : {feuille}")

    # ===== Base : bloc 1 =====
    df_global = resultats_bloc1[feuille]["df"][
        ["Numero_inclusion", "Condition", "rejeter_bloc1"]
    ].copy()

    # ===== Fusion bloc 2S =====
    df_global = df_global.merge(
        resultats_bloc2_S[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc2S"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Fusion bloc 2R =====
    df_global = df_global.merge(
        resultats_bloc2_R[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc2R"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Fusion bloc 3 =====
    df_global = df_global.merge(
        resultats_bloc3[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc3"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Affichage debug AVANT filtre =====
    print("\n📊 df_global AVANT fillna et filtres :")
    display(df_global)

    # ===== Sécurité : NaN → rejet (2) =====
    df_global[colonnes_blocs] = df_global[colonnes_blocs].fillna(2)

    # ===== Filtrage conditionnel S / R =====
    mask_S = (
        (df_global["Condition"] == "Standard") &
        (df_global["rejeter_bloc1"] == 0) &
        (df_global["rejeter_bloc2S"] == 0) &
        (df_global["rejeter_bloc3"] == 0)
    )

    mask_R = (
        (df_global["Condition"] == "Réduction de la menace") &
        (df_global["rejeter_bloc1"] == 0) &
        (df_global["rejeter_bloc2R"] == 0) &
        (df_global["rejeter_bloc3"] == 0)
    )

    mask_valide = mask_S | mask_R

    # ===== Extraction =====
    candidats_valides = df_global.loc[
        mask_valide, "Numero_inclusion"
    ].tolist()

    candidats_rejetes = df_global.loc[
        ~mask_valide, "Numero_inclusion"
    ].tolist()

    # ===== Résumés =====
    print(f"\nNombre total de candidats : {len(df_global)}")
    print(f"Candidats VALIDES (selon Condition) : {len(candidats_valides)}")
    print(f"Candidats REJETÉS : {len(candidats_rejetes)}")

    print("\nCandidats valides par condition :")
    print(df_global.loc[mask_valide, "Condition"].value_counts())

    print("\nListe des candidats valides :")
    for pid in candidats_valides:
        print(f" - {pid}")

    # ===== Stockage =====
    resultats_globaux[feuille] = {
        "df": df_global,
        "valides": candidats_valides,
        "rejetes": candidats_rejetes,
        "n_valides": len(candidats_valides),
        "n_rejetes": len(candidats_rejetes),
    }


Analyse globale – Feuille : APHM

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,Condition,rejeter_bloc1,rejeter_bloc2S,rejeter_bloc2R,rejeter_bloc3
0,0101CAR,Réduction de la menace,0,0,0,0
1,0102PCR,Réduction de la menace,0,0,0,0
2,0103SHS,Standard,0,0,2,0
3,0105PNR,Réduction de la menace,0,0,0,0
4,0104FJS,Standard,0,0,2,0
5,0106JLS,Standard,0,0,2,0
6,0107DSS,Standard,0,0,2,0
7,0108BFS,Standard,0,0,2,0
8,0109GSS,Standard,0,0,2,0
9,0110LPR,Réduction de la menace,0,0,0,0



Nombre total de candidats : 25
Candidats VALIDES (selon Condition) : 24
Candidats REJETÉS : 1

Candidats valides par condition :
Condition
Réduction de la menace    13
Standard                  11
Name: count, dtype: int64

Liste des candidats valides :
 - 0101CAR
 - 0102PCR
 - 0103SHS
 - 0105PNR
 - 0104FJS
 - 0106JLS
 - 0107DSS
 - 0108BFS
 - 0109GSS
 - 0110LPR
 - 0111MNR
 - 0112BSR
 - 0113BMR
 - 0114LMS
 - 0115CJR
 - 0116VAR
 - 0117POS
 - 0118TNS
 - 0119LJS
 - 0120LCR
 - 0121MJR
 - 0123RMS
 - 0124BAR
 - 0125BMR

Analyse globale – Feuille : CAEN

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,Condition,rejeter_bloc1,rejeter_bloc2S,rejeter_bloc2R,rejeter_bloc3
0,0301GNR,Réduction de la menace,0,0,0,0
1,0302ZMR,Réduction de la menace,0,0,0,0
2,0303PAR,Réduction de la menace,0,2,0,0
3,0304VMR,Réduction de la menace,0,0,0,0
4,0305LCR,Réduction de la menace,0,2,0,0
5,0307SMR,Réduction de la menace,0,0,0,0
6,0306MDR,Réduction de la menace,0,0,0,0
7,0308RGR,Réduction de la menace,0,0,0,0
8,0309DBS,Standard,0,0,2,0
9,0311CJS,Standard,0,0,2,0



Nombre total de candidats : 56
Candidats VALIDES (selon Condition) : 47
Candidats REJETÉS : 9

Candidats valides par condition :
Condition
Réduction de la menace    24
Standard                  23
Name: count, dtype: int64

Liste des candidats valides :
 - 0301GNR
 - 0302ZMR
 - 0303PAR
 - 0304VMR
 - 0305LCR
 - 0307SMR
 - 0306MDR
 - 0308RGR
 - 0309DBS
 - 0311CJS
 - 0310ACS
 - 0315VCS
 - 0318PMS
 - 0313FPR
 - 0312JMS
 - 0316TMS
 - 0317LGS
 - 0319LJS
 - 0320DGS
 - 0321DRR
 - 0323RFS
 - 0325LCR
 - 0327IAR
 - 0326BJR
 - 0328MPR
 - 0330RTR
 - 0329NMR
 - 0331GRS
 - 0332MCR
 - 0333MMS
 - 0334TVS
 - 0336CBS
 - 0337BFS
 - 0339NPR
 - 0341LJS
 - 0342VNS
 - 0343PAS
 - 0349LLS
 - 0347DMR
 - 0346GLS
 - 0348GCR
 - 0350MYR
 - 0351FIR
 - 0352RNR
 - 0354GKR
 - 0355BAS
 - 0356DMS

Analyse globale – Feuille : POITIERS

📊 df_global AVANT fillna et filtres :


...


Nombre total de candidats : 8
Candidats VALIDES (selon Condition) : 8
Candidats REJETÉS : 0

Candidats valides par condition :
Condition
Réduction de la menace    4
Standard                  4
Name: count, dtype: int64

Liste des candidats valides :
 - 0801HDR
 - 0802LAS
 - 0804GOR
 - 0803DPS
 - 0805BMS
 - 0806KHS
 - 0807OMR
 - 0808PJR

Analyse globale – Feuille : CGD

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,Condition,rejeter_bloc1,rejeter_bloc2S,rejeter_bloc2R,rejeter_bloc3
0,0901SMR,Réduction de la menace,0,0,0,0
1,0903DJS,Standard,2,0,2,0
2,0902SIS,Standard,0,0,2,0
3,0906DNR,Réduction de la menace,0,2,0,0
4,0904KJS,Standard,0,0,2,0
5,0905SLR,Réduction de la menace,0,2,0,0



Nombre total de candidats : 6
Candidats VALIDES (selon Condition) : 5
Candidats REJETÉS : 1

Candidats valides par condition :
Condition
Réduction de la menace    3
Standard                  2
Name: count, dtype: int64

Liste des candidats valides :
 - 0901SMR
 - 0902SIS
 - 0906DNR
 - 0904KJS
 - 0905SLR

Analyse globale – Feuille : CERCA

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,Condition,rejeter_bloc1,rejeter_bloc2S,rejeter_bloc2R,rejeter_bloc3
0,0401BDS,Standard,2,2,2,2
1,0402TFR,Réduction de la menace,2,2,2,2
2,0404DER,Réduction de la menace,2,2,2,2
3,0405RCR,Réduction de la menace,2,2,2,2
4,0406BBS,Standard,2,2,2,2
5,0407HMS,Standard,2,2,2,2
6,0408SJS,Standard,2,2,2,2
7,0409HCR,Réduction de la menace,2,2,2,2
8,0410FMS,Standard,2,2,2,2
9,0411NPR,Réduction de la menace,2,2,2,2



Nombre total de candidats : 40
Candidats VALIDES (selon Condition) : 11
Candidats REJETÉS : 29

Candidats valides par condition :
Condition
Réduction de la menace    7
Standard                  4
Name: count, dtype: int64

Liste des candidats valides :
 - 0413MMR
 - 0414PVR
 - 0415HJS
 - 0417FAS
 - 0417KMS
 - 0418MPR
 - 0420MCR
 - 0420LMR
 - 0421ACS
 - 0423RMR
 - 0424BLR


In [12]:
total_valides = sum(
    res["n_valides"] for res in resultats_globaux.values()
)

print("=" * 70)
print(f"✅ Nombre TOTAL de candidats valides (toutes feuilles confondues) : {total_valides}")

✅ Nombre TOTAL de candidats valides (toutes feuilles confondues) : 165


# 165 candidats retenus

# Recherche des candidats et téléchargement de leur fichier V2 BB RR

In [13]:
import pandas as pd

# Afficher toutes les lignes
pd.set_option('display.max_rows', None)

# Afficher toutes les colonnes
pd.set_option('display.max_columns', None)

# Afficher toute la largeur de chaque colonne
pd.set_option('display.max_colwidth', None)

## APHM

In [14]:
# ===== Racine APHM =====
racine_aphm = r"C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran"

# ===== Candidats valides =====
candidats_aphm = [c.upper() for c in resultats_globaux["APHM"]["valides"]]

# ===== Mapping PID → Condition =====
df_conditions = resultats_globaux["APHM"]["df"][[
    "Numero_inclusion",
    "Condition"
]].copy()

df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()
df_conditions["Condition"] = df_conditions["Condition"].astype(str)

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

rows = []

for root, dirs, files in os.walk(racine_aphm):

    if "V2" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== détection PID dans tout le chemin =====
        pid_trouve = None
        for pid in candidats_aphm:
            if pid in chemin_upper:
                pid_trouve = pid
                break

        if not pid_trouve:
            continue

        # ===== récupération de la condition =====
        condition = dict_condition.get(pid_trouve)

        if condition is None:
            print(f"⚠️ Condition manquante pour {pid_trouve} → ignoré")
            continue

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Condition": condition,
            "Feuille": "APHM"
        })

# ===== DataFrame brut =====
df_aphm = pd.DataFrame(rows)

# ===== Suppression doublons =====
df_aphm_final = (
    df_aphm
    .assign(
        priorite_ceinture=df_aphm["Chemin"].str.contains("CEINTURE", case=False)
    )
    .sort_values(
        ["Numero_inclusion", "priorite_ceinture"],
        ascending=[True, False]
    )
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .drop(columns="priorite_ceinture")
    .reset_index(drop=True)
)

print(f"Candidats attendus APHM : {len(candidats_aphm)}")
print(f"Fichiers BB/BR_RR APHM V2 retenus : {len(df_aphm_final)}")

display(df_aphm_final)


Candidats attendus APHM : 24
Fichiers BB/BR_RR APHM V2 retenus : 24


,Numero_inclusion,Chemin,Condition,Feuille
0,0101CAR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0101CAR\0101CAR 06-07-2018 V2\Ceinture 0101CAR\2018_07_06-08_24_06_BB_RR.csv,Réduction de la menace,APHM
1,0102PCR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0102PCR\0102PCR 09-07-2018 V2\Ceinture 0102PCR\2018_07_09-08_57_05_BB_RR.csv,Réduction de la menace,APHM
2,0103SHS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0103SHS\0103SHS 10-09-2018 V2\Ceinture 0103SHS\2018_09_10-09_03_59_BB_RR.csv,Standard,APHM
3,0104FJS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0104FJS\0104FJS 29-11-2018 V2\Ceinture 0104FJS\2018_11_29-08_27_38_BB_RR.csv,Standard,APHM
4,0105PNR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0105PNR\0105PNR 10-12-2018 V2\Ceinture 0105PNR\2018_12_10-08_23_34_BB_RR.csv,Réduction de la menace,APHM
5,0106JLS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0106JLS\0106JLS 03-01-2019 V2\Ceinture 0106JLS\2019_01_03-08_25_39_BB_RR.csv,Standard,APHM
6,0107DSS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0107DSS\0107DSS 14-01-2019 V2\Ceinture 0107DSS\2019_01_14-08_24_42_BB_RR.csv,Standard,APHM
7,0108BFS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0108BFS\0108BFS 17-01-2019 V2\Ceinture 0108BFS\2019_01_17-08_42_57_BB_RR.csv,Standard,APHM
8,0109GSS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0109GSS\0109GSS 18-02-2019 V2\Ceinture 0109GSS\2019_02_18-08_56_33_BB_RR.csv,Standard,APHM
9,0110LPR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0110LPR\0110LPR 26-02-2019 V2\Ceinture 0110LPR\2019_02_26-08_44_39_BB_RR.csv,Réduction de la menace,APHM


## CAEN

In [15]:
# ===== Racine et candidats CAEN =====
racine_caen = r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN"
candidats_caen = [c.upper() for c in resultats_globaux["CAEN"]["valides"]]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["CAEN"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

# ===== Paths forcés pour candidats avec pattern différent =====
paths_forces = {
    "0302ZMR": r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0302ZMR\ceinture\2019_01_24-09_37_03 0302ZMR\2019_01_24-09_37_03_BB_RR.csv",
    "0304VMR": r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0304VMR\Ceinture\2019_03_25-09_49_47 0304VMR\2019_03_25-09_49_47_BB_RR.csv",
    "0334TVS": r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2020_10_01_09_40_22_0334TVS\2020_10_01__09_40_22_BR_RR.csv",
    "0339NPR": r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2021_02_25_09_34_13 0339PNR\2021_02_25__09_34_13_BR_RR.csv",
}

# ===== Pattern PID dans le chemin (hors paths forcés) =====
patterns_candidats = {
    pid: re.compile(rf"\b{pid}\b", re.IGNORECASE)
    for pid in candidats_caen if pid not in paths_forces
}

rows = []

# ===== Parcours des fichiers =====
for root, dirs, files in os.walk(racine_caen):
    for f in files:
        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)

        for pid, pat in patterns_candidats.items():
            if pat.search(chemin):
                rows.append({
                    "Numero_inclusion": pid,
                    "Chemin": chemin,
                    "Condition": dict_condition.get(pid),
                    "Feuille": "CAEN"
                })
                break

# ===== Ajout des candidats avec path forcé =====
for pid, chemin in paths_forces.items():
    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin,
        "Condition": dict_condition.get(pid),
        "Feuille": "CAEN"
    })

# ===== Suppression doublons =====
df_caen = pd.DataFrame(rows)

df_caen_final = (
    df_caen
    .assign(
        priorite_ceinture=df_caen["Chemin"].str.contains("ceinture", case=False)
    )
    .sort_values(
        ["Numero_inclusion", "priorite_ceinture"],
        ascending=[True, False]
    )
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .drop(columns="priorite_ceinture")
    .reset_index(drop=True)
)

# ===== Résumé =====
print(f"Candidats attendus CAEN : {len(candidats_caen)}")
print(f"Fichiers BB/BR_RR CAEN retenus : {len(df_caen_final)}")

display(df_caen_final)

# pour le 0323RFS il s'agit de la v4, supprimer plus tard

Candidats attendus CAEN : 47
Fichiers BB/BR_RR CAEN retenus : 47


,Numero_inclusion,Chemin,Condition,Feuille
0,0301GNR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0301GNR\ceinture\2019_01_16-10_11_49 0301GNR\2019_01_16-10_11_49_BB_RR.csv,Réduction de la menace,CAEN
1,0302ZMR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0302ZMR\ceinture\2019_01_24-09_37_03 0302ZMR\2019_01_24-09_37_03_BB_RR.csv,Réduction de la menace,CAEN
2,0303PAR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0303PAR\Ceinture\2019_03_14-09_58_28 0303PAR\2019_03_14-09_58_28_BB_RR.csv,Réduction de la menace,CAEN
3,0304VMR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\fichiers ancien serveur\0304VMR\Ceinture\2019_03_25-09_49_47 0304VMR\2019_03_25-09_49_47_BB_RR.csv,Réduction de la menace,CAEN
4,0305LCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2019_04_25-10_01_08 0305LCR\2019_04_25-10_01_08_BB_RR.csv,Réduction de la menace,CAEN
5,0306MDR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2019_05_06-14_17_53 0306MDR\2019_05_06-14_17_53_BB_RR.csv,Réduction de la menace,CAEN
6,0307SMR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2019_05_13-10_23_50 0307SMR\2019_05_13-10_23_50_BB_RR (1).csv,Réduction de la menace,CAEN
7,0308RGR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2019_06_25-10_55_55 0308RGR\2019_06_25-10_55_55_BB_RR.csv,Réduction de la menace,CAEN
8,0309DBS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2019_06_27-13_19_31 0309DBS\2019_06_27-13_19_31_BB_RR.csv,Standard,CAEN
9,0310ACS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-CAEN\ALLAIN\ALLAIN\CEINTURE\2019_07_03-09_56_22 0310ACS\2019_07_03-09_56_22_BB_RR.csv,Standard,CAEN


## POITIERS

In [16]:
# ===== Racine POITIERS =====
racine_poitiers = r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY"

# ===== Candidats valides =====
candidats_poitiers = [c.upper() for c in resultats_globaux["POITIERS"]["valides"]]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["POITIERS"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

# ===== Regex POITIERS (TRONQUÉE → clé pour 0408) =====
# ex : 0408BCS → match "0408 BC"
patterns_candidats = {
    pid: re.compile(
        rf"{pid[:4]}\s*-?\s*{pid[4:6]}",
        re.IGNORECASE
    )
    for pid in candidats_poitiers
}

fichiers_par_candidat = defaultdict(list)

for root, dirs, files in os.walk(racine_poitiers):

    if "V2" not in root.upper():
        continue

    for f in files:
        if not f.lower().endswith(".csv"):
            continue

        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)

        for pid, pat in patterns_candidats.items():
            if pat.search(chemin):
                fichiers_par_candidat[pid].append(chemin)
                break

# ===== Sélection finale + DataFrame =====
rows = []

for pid, fichiers in fichiers_par_candidat.items():

    fichiers = sorted(set(fichiers))  # sécurité doublons exacts

    # priorité CEINTURE
    ceinture = [f for f in fichiers if "ceinture" in f.lower()]
    chemin_final = ceinture[0] if ceinture else fichiers[0]

    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin_final,
        "Condition": dict_condition.get(pid),
        "Feuille": "POITIERS"
    })

df_poitiers_final = pd.DataFrame(rows)

# ===== Vérifications =====
print(f"Candidats attendus POITIERS : {len(candidats_poitiers)}")
print(f"Fichiers BB/BR_RR POITIERS retenus : {len(df_poitiers_final)}")

manquants = sorted(set(candidats_poitiers) - set(df_poitiers_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats POITIERS sont présents")

display(df_poitiers_final)


Candidats attendus POITIERS : 13
Fichiers BB/BR_RR POITIERS retenus : 13
✅ Tous les candidats POITIERS sont présents


,Numero_inclusion,Chemin,Condition,Feuille
0,0401TSS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0401 TS\0401TSS - V2\0401TSS ceinture\No Team Assigned\2019_04_05-09_52_46\2019_04_05-09_52_46_BB_RR.csv,Standard,POITIERS
1,0402LLS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0402 LL\0402LLS- V2\0402LLS- ceinture\No Team Assigned\2019_04_12-11_01_28\2019_04_12-11_01_28_BB_RR.csv,Standard,POITIERS
2,0403DCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0403 DC\0403DCR-V2\0403DCR- ceinture\No Team Assigned\2019_05_24-09_36_16\2019_05_24-09_36_16_BB_RR.csv,Réduction de la menace,POITIERS
3,0404CYS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0404 CY\0404CYS-V2\0404YC-ceinture\2019_07_19-10_32_55\2019_07_19-10_32_55_BB_RR.csv,Standard,POITIERS
4,0405FCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0405 FC\0405 FCR-V2\0405FCR- ceinture\No Team Assigned\2019_07_04-09_33_11\2019_07_04-09_33_11_BB_RR.csv,Réduction de la menace,POITIERS
5,0407LJR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0407 LJ\0407LJR-V2\0407LJ-ceinture\No Team Assigned\2019_09_19-09_38_39\2019_09_19-09_38_39_BB_RR.csv,Réduction de la menace,POITIERS
6,0408BCS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0408 BC\0408 BC V2\0408BC-ceinture\No Team Assigned\2019_11_22-09_28_20\2019_11_22-09_28_20_BB_RR.csv,Standard,POITIERS
7,0409PHR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0409 PH\0409 PH- V2\0409PHR ceinture\No Team Assigned\2019_11_29-09_34_34\2019_11_29-09_34_34_BB_RR.csv,Réduction de la menace,POITIERS
8,0410PGS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0410 PG\0410 PGS - V2\ceinture\No Team Assigned\2020_01_09-10_03_29\2020_01_09-10_03_29_BB_RR.csv,Standard,POITIERS
9,0411VES,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0411 VE\0411 VES-V2\0411VE ceinture\No Team Assigned\2020_02_20-09_42_09\2020_02_20-09_42_09_BB_RR.csv,Standard,POITIERS


## ROUEN

In [17]:
# ===== Racine ROUEN =====
racine_rouen = r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER"

# ===== Candidats valides =====
candidats_rouen = [c.upper() for c in resultats_globaux["ROUEN"]["valides"]]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["ROUEN"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

# ===== paramètres poids =====
POIDS_MIN = 3000 * 1024
POIDS_MAX = 8000 * 1024

# ===== pattern numéro (ex : 05-12) =====
patterns_candidats = [
    (pid, re.compile(rf"{pid[:2]}-{pid[2:4]}", re.IGNORECASE))
    for pid in candidats_rouen
]

# ===== pattern date =====
date_pattern = re.compile(r"(\d{4}_\d{2}_\d{2}[-_]\d{2}_\d{2}_\d{2})")

# ===== collecte brute =====
fichiers_dict = defaultdict(list)

for root, dirs, files in os.walk(racine_rouen):

    if "V2" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue

        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)

        # ---- filtre poids pour eviter les doublons ----
        try:
            taille = os.path.getsize(chemin)
        except OSError:
            continue

        if not (POIDS_MIN <= taille <= POIDS_MAX):
            continue

        # ---- identification candidat ----
        pid_trouve = None
        for pid, pat in patterns_candidats:
            if pat.search(chemin):
                pid_trouve = pid
                break

        if not pid_trouve:
            continue

        # ---- date ----
        m = date_pattern.search(chemin)
        date_str = m.group(1) if m else "inconnue"

        fichiers_dict[(pid_trouve, date_str)].append(chemin)

# ===== Sélection finale + DataFrame =====
rows = []

for (pid, date_str), fichiers in fichiers_dict.items():

    fichiers = sorted(set(fichiers)) 
    chemin_final = fichiers[0]

    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin_final,
        "Condition": dict_condition.get(pid),
        "Feuille": "ROUEN"
    })

df_rouen_final = pd.DataFrame(rows)

# ===== Résumé & contrôle =====
print(f"Candidats attendus ROUEN : {len(candidats_rouen)}")
print(f"Fichiers BB/BR_RR ROUEN retenus : {len(df_rouen_final)}")

manquants = sorted(set(candidats_rouen) - set(df_rouen_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats ROUEN manquants :", manquants)
else:
    print("✅ Tous les candidats ROUEN sont présents")

display(df_rouen_final)

# 0515 FAR date et heure correspondent pas et temps d'experience ne correspond pas non plus a la durée excel : on le rejette

Candidats attendus ROUEN : 14
Fichiers BB/BR_RR ROUEN retenus : 12
⚠️ Candidats ROUEN manquants : ['0501MBS', '0503LCR']


,Numero_inclusion,Chemin,Condition,Feuille
0,0502LIR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-02-LI-R\V2\CEINTURE\2019_02_28-09_42_09_BB_RR.csv,Réduction de la menace,ROUEN
1,0504BCS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-04-BC-S\V2\ceinture 05-04-BC-S\2019_04_23-09_59_09\2019_04_23-09_59_09_BB_RR.csv,Standard,ROUEN
2,0505PPR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-05-PP-R\V2\CEINTURE\2019_05_23-09_48_02_BB_RR.csv,Réduction de la menace,ROUEN
3,0506VMS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-06-VM-S\V2\CEINTURE 05-06-VM-S\2019_05_27-09_37_10_BB_RR.csv,Standard,ROUEN
4,0507BDS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-07-BD-S\V2\CEINTURE\2019_06_24-09_49_35_BB_RR.csv,Standard,ROUEN
5,0508SRR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-08-SR-R\V2\CEINTURE V2\2019_07_25-09_37_55_BB_RR.csv,Réduction de la menace,ROUEN
6,0509LGS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-09-LC-S\V2\CEINTURE\2019_10_17-09_30_14_BB_RR.csv,Standard,ROUEN
7,0510DFS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-10-DF-S\V2\CEINTURE\2020_02_20-09_49_46_BB_RR.csv,Standard,ROUEN
8,0511LCS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-11-LC-S\05-11-LC-S-V2\CEINTURE\2021_01_07-10_11_23_BB_RR.csv,Standard,ROUEN
9,0512RCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-ROUEN\HANNIER\HANNIER\05-12-RC-R\V2\CEINTURE\Record 2_2021_05_04-08_47_59_BB_RR (1).csv,Réduction de la menace,ROUEN


## LAVERAN

In [18]:
# ===== Racine LAVERAN =====
racine_laveran = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran"
)

# ===== Candidats attendus =====
candidats_laveran = [c.upper() for c in resultats_globaux["LAVERAN"]["valides"]]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["LAVERAN"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

rows = []

for root, dirs, files in os.walk(racine_laveran):

    if "V2" not in root.upper():
        continue

    for f in files:

        # CSV BB/BR_RR uniquement
        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== identification du candidat =====
        pid = None
        for c in candidats_laveran:
            if c in chemin_upper:
                pid = c
                break

        if not pid:
            continue

        rows.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Condition": dict_condition.get(pid),
            "Feuille": "LAVERAN"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================

df_laveran_final = pd.DataFrame(rows)

df_laveran_final = (
    df_laveran_final
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================

print("\n====================================")
print(f"Candidats attendus LAVERAN : {len(candidats_laveran)}")
print(f"Fichiers retenus LAVERAN    : {len(df_laveran_final)}")

manquants = sorted(
    set(candidats_laveran) - set(df_laveran_final["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats LAVERAN manquants :", manquants)
else:
    print("✅ Tous les candidats LAVERAN sont présents")

display(df_laveran_final)


OK [0626MCS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0626MCS\V2\ceinture\2020_01_21-09_12_01\2020_01_21-09_12_01_BB_RR.csv
OK [0628DMR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0628DMR\V2\ceinture\2020_01_30-10_11_25\2020_01_30-10_11_25_BB_RR.csv
OK [0629TCR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0629TCR\V2\ceinture\No Team Assigned\2020_02_04-09_40_12\2020_02_04-09_40_12_BB_RR.csv
OK [0630RJR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0630RJR\0630RJR_V2\ceinture\No Team Assigned\2020_02_20-10_24_36\2020_02_20-10_24_36_BB_RR.csv
OK [0631LHR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0631LHR\V2\ceinture\2020_02_25-10_54_51\2020_02_25-10_54_51_BB_RR.csv

Candidats attendus LAVERAN 

,Numero_inclusion,Chemin,Condition,Feuille
0,0626MCS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0626MCS\V2\ceinture\2020_01_21-09_12_01\2020_01_21-09_12_01_BB_RR.csv,Standard,LAVERAN
1,0628DMR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0628DMR\V2\ceinture\2020_01_30-10_11_25\2020_01_30-10_11_25_BB_RR.csv,Réduction de la menace,LAVERAN
2,0629TCR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0629TCR\V2\ceinture\No Team Assigned\2020_02_04-09_40_12\2020_02_04-09_40_12_BB_RR.csv,Réduction de la menace,LAVERAN
3,0630RJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0630RJR\0630RJR_V2\ceinture\No Team Assigned\2020_02_20-10_24_36\2020_02_20-10_24_36_BB_RR.csv,Réduction de la menace,LAVERAN
4,0631LHR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0631LHR\V2\ceinture\2020_02_25-10_54_51\2020_02_25-10_54_51_BB_RR.csv,Réduction de la menace,LAVERAN


## LPC

In [19]:
# ===== Racine LPC =====
racine_lpc = r"C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging"

# ===== Candidats attendus =====
candidats_lpc = [c.upper() for c in resultats_globaux["LPC"]["valides"]]

# ==========================================================
# CORRECTION MANUELLE DES PID (Excel → Arborescence)
# ==========================================================
CORRESPONDANCE_IDS = {
    "0117MSR": "0117MMR"
}

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["LPC"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

# ===== paramètres poids =====
POIDS_MIN = 2000 * 1024   # 2 Mo
POIDS_MAX = 8000 * 1024   # 8 Mo

# ===== date dans nom de fichier =====
date_pattern = re.compile(r"(\d{4}_\d{2}_\d{2}-\d{2}_\d{2}_\d{2})")

# ===== collecte brute =====
fichiers_dict = defaultdict(list)

for root, dirs, files in os.walk(racine_lpc):

    if "V2" not in root.upper():
        continue

    for f in files:

        # CSV BB/BR_RR uniquement
        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ---- filtre poids ----
        try:
            taille = os.path.getsize(chemin)
        except OSError:
            continue

        if not (POIDS_MIN <= taille <= POIDS_MAX):
            continue

        # ---- identification candidat ----
        pid = None
        for c in candidats_lpc:

            # ID réel à chercher dans l'arborescence
            id_recherche = CORRESPONDANCE_IDS.get(c, c)

            if id_recherche in chemin_upper:
                pid = c  
                break

        if not pid:
            continue

        # ---- date ----
        m = date_pattern.search(f)
        date_str = m.group(1) if m else "inconnue"

        fichiers_dict[(pid, date_str)].append(chemin)

# ===== DataFrame + Suppression des doublons =====
rows = []

for (pid, date_str), fichiers in fichiers_dict.items():

    fichiers = sorted(set(fichiers))
    chemin_final = fichiers[0]

    rows.append({
        "Numero_inclusion": pid,  # toujours ID Excel
        "Chemin": chemin_final,
        "Condition": dict_condition.get(pid),
        "Feuille": "LPC"
    })

df_lpc_final = pd.DataFrame(rows)

# ===== Affichage =====
print(f"Candidats attendus LPC : {len(candidats_lpc)}")
print(f"Fichiers BB/BR_RR retenus : {len(df_lpc_final)}")

manquants = sorted(
    set(candidats_lpc) - set(df_lpc_final["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats LPC manquants :", manquants)
else:
    print("✅ Tous les candidats LPC sont présents")

display(df_lpc_final)

Candidats attendus LPC : 38
Fichiers BB/BR_RR retenus : 25
⚠️ Candidats LPC manquants : ['0132GAR', '0133BPS', '0134IFR', '0135KLS', '0136CJR', '0137BMS', '0138NWR', '0139BMR', '0140LMR', '0141HJR', '0142LLS', '0143EBR', '0144ZGR']


,Numero_inclusion,Chemin,Condition,Feuille
0,0101EMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\V2\Ceinture\2019_01_22-10_10_43\2019_01_22-10_10_43_BB_RR.csv,Standard,LPC
1,0103BPS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V2\Ceinture\2018_12_11-10_08_18\2018_12_11-10_08_18_BB_RR.csv,Standard,LPC
2,0104IBS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0104IBS\V2\Ceinture\2018_11_28-09_57_59\2018_11_28-09_57_59_BB_RR.csv,Standard,LPC
3,0106DJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0106DJR\V2\Ceinture\2018_12_19-10_05_30\2018_12_19-10_05_30_BB_RR.csv,Réduction de la menace,LPC
4,0108MMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0108MMS\V2\Ceinture\2019_01_24-09_57_32_BB_RR.csv,Standard,LPC
5,0109MJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0109MJR\V2\Ceinture\2018_12_13-09_37_28_BB_RR.csv,Réduction de la menace,LPC
6,0110RMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0110RMS\V2\Ceinture\2018_12_14-09_38_18_BB_RR.csv,Standard,LPC
7,0112DRR,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0112DRR\V2\Ceinture\No Team Assigned\2019_02_06-09_34_41\2019_02_06-09_34_41_BB_RR.csv,Réduction de la menace,LPC
8,0113DGR,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0113DGR\V2\Ceinture\2018_11_27-09_37_54_BB_RR.csv,Réduction de la menace,LPC
9,0114MLR,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0114MLR\V2\Ceinture\2019_01_07-09_59_01_BB_RR.csv,Réduction de la menace,LPC


In [20]:
# ===== Racines =====
racines = [
    r"C:\Users\judupont\Desktop\data_aging_11_2025_copie\LNSC\LNSC\DESHAYES\DATA_Ancillaire",
    r"C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo"
]

# ===== Candidats à garder =====
candidats_a_garder = [
    c.upper() for c in resultats_globaux["LPC"]["valides"][25:]
]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["LPC"]["df"][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

# ===== Paramètres poids =====
POIDS_MIN = 2000 * 1024   # 2 Mo
POIDS_MAX = 8000 * 1024   # 8 Mo

# ===== Collecte DataFrame =====
rows = []

for racine in racines:
    print(f"\n--- Scan de : {racine}")

    for root, dirs, files in os.walk(racine):

        if "V2" not in root.upper():
            continue

        for f in files:

            if not f.lower().endswith(".csv"):
                continue
            if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
                continue

            chemin = os.path.join(root, f)

            # ---- filtre poids ----
            try:
                taille = os.path.getsize(chemin)
            except OSError:
                continue

            if not (POIDS_MIN <= taille <= POIDS_MAX):
                continue

            # ---- identification candidat ----
            pid = None
            chemin_upper = chemin.upper()
            for c in candidats_a_garder:
                if c in chemin_upper:
                    pid = c
                    break

            if not pid:
                continue

            rows.append({
                "Numero_inclusion": pid,
                "Chemin": chemin,
                "Condition": dict_condition.get(pid),
                "Feuille": "LPC"
            })

            print(f"OK [{pid}] :", chemin)

# ===== Création DataFrame =====
df_lpc2 = pd.DataFrame(rows)

# ===== Déduplication =====
df_lpc2 = df_lpc2.drop_duplicates(
    subset="Numero_inclusion",
    keep="first"
).reset_index(drop=True)

# ===== Résumé & contrôles =====
print("\n====================================")
print(f"Candidats attendus LPC (partie 2) : {len(candidats_a_garder)}")
print(f"Fichiers retenus                  : {len(df_lpc2)}")

manquants = sorted(
    set(candidats_a_garder) - set(df_lpc2["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats LPC manquants :", manquants)
else:
    print("✅ Tous les candidats LPC sont présents")

display(df_lpc2)



--- Scan de : C:\Users\judupont\Desktop\data_aging_11_2025_copie\LNSC\LNSC\DESHAYES\DATA_Ancillaire

--- Scan de : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo
OK [0132GAR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0132GAR\V2\Ceinture\2020_02_19-09_57_21\2020_02_19-09_57_21_BB_RR.csv
OK [0133BPS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0133BPS-0157BPS\V2\Ceinture\No Team Assigned\2020_02_26-10_04_01\2020_02_26-10_04_01_BB_RR.csv
OK [0134IFR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0134IFR\V2\ceinture\No Team Assigned\2020_03_03-09_35_14\2020_03_03-09_35_14_BB_RR.csv
OK [0135KLS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0135KLS\0135KLS_0163KLS_V2\ceinture\No Team Ass

,Numero_inclusion,Chemin,Condition,Feuille
0,0132GAR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0132GAR\V2\Ceinture\2020_02_19-09_57_21\2020_02_19-09_57_21_BB_RR.csv,Réduction de la menace,LPC
1,0133BPS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0133BPS-0157BPS\V2\Ceinture\No Team Assigned\2020_02_26-10_04_01\2020_02_26-10_04_01_BB_RR.csv,Standard,LPC
2,0134IFR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0134IFR\V2\ceinture\No Team Assigned\2020_03_03-09_35_14\2020_03_03-09_35_14_BB_RR.csv,Réduction de la menace,LPC
3,0135KLS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0135KLS\0135KLS_0163KLS_V2\ceinture\No Team Assigned\2021_03_03-08_43_04\2021_03_03-08_43_04_BB_RR.csv,Standard,LPC
4,0136CJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0136CJR\0136CJR-V2\CEINTURE\No Team Assigned\2021_04_02-09_00_53\2021_04_02-09_00_53_BB_RR.csv,Réduction de la menace,LPC
5,0137BMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0137BMS-V2\CEINTURE\No Team Assigned\2021_04_12-09_58_17\2021_04_12-09_58_17_BB_RR (1).csv,Standard,LPC
6,0138NWR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0138NWR\0138NWR-V2\ceinture\No Team Assigned\2021_06_02-10_11_00\2021_06_02-10_11_00_BB_RR.csv,Réduction de la menace,LPC
7,0144ZGR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0144ZGR\0144ZGR_V2\ceinture\No Team Assigned\2022_06_29-10_04_46\2022_06_29-10_04_46_BB_RR.csv,Réduction de la menace,LPC
8,0140LMR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0168LMR 0140LMR\V2 0140LMR\CEINTURE\No Team Assigned\2022_01_13-09_40_17\2022_01_13-09_40_17_BB_RR.csv,Réduction de la menace,LPC
9,0143EBR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0171EBR _ 0143EBR\0143EBR V2\ceinture\No Team Assigned\2022_04_27-11_02_28\2022_04_27-11_02_28_BB_RR.csv,Réduction de la menace,LPC


## SAINTE-MARGUERITE

In [21]:
# ===== Racine SAINTE-MARGUERITE =====
racine_st_marguerite = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite"
)

# ===== Candidats attendus =====
candidats_sm = [
    c.upper() for c in resultats_globaux["SAINTE-MARGUERITE"]["valides"]
]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["SAINTE-MARGUERITE"]["df"][
    ["Numero_inclusion", "Condition"]
].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

rows_sm = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_st_marguerite):

    if "V2" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== identification du candidat =====
        pid = None
        for c in candidats_sm:
            if c in chemin_upper:
                pid = c
                break

        if not pid:
            continue

        rows_sm.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Condition": dict_condition.get(pid),
            "Feuille": "SAINTE-MARGUERITE"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_sm = pd.DataFrame(rows_sm)

df_sm = (
    df_sm
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus SAINTE-MARGUERITE : {len(candidats_sm)}")
print(f"Fichiers retenus                     : {len(df_sm)}")

manquants = sorted(
    set(candidats_sm) - set(df_sm["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats SAINTE-MARGUERITE sont présents")

display(df_sm)


OK [0801HDR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0801HDR\0801HDR - V2\CEINTURE_0801HDR_V2\No Team Assigned\2022_03_23-09_34_20\2022_03_23-09_34_20_BB_RR.csv
OK [0802LAS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0802LAS\0802LAS - V2\CEINTURE_0802LAS_V2\No Team Assigned\2022_03_21-15_42_25\2022_03_21-15_42_25_BB_RR.csv
OK [0803DPS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0803DPS\0803DPS - V2\CEINTURE_0803DPS_V2\No Team Assigned\2022_03_30-09_33_44\2022_03_30-09_33_44_BB_RR.csv
OK [0804GOR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0804GOR\0804GOR - V2\CEINTURE_0804GOR_V2\No Team Assigned\2022_04_13-09_22_46\2022_04_13-09_22_46_BB_RR.csv
OK [0805BMS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATI

,Numero_inclusion,Chemin,Condition,Feuille
0,0801HDR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0801HDR\0801HDR - V2\CEINTURE_0801HDR_V2\No Team Assigned\2022_03_23-09_34_20\2022_03_23-09_34_20_BB_RR.csv,Réduction de la menace,SAINTE-MARGUERITE
1,0802LAS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0802LAS\0802LAS - V2\CEINTURE_0802LAS_V2\No Team Assigned\2022_03_21-15_42_25\2022_03_21-15_42_25_BB_RR.csv,Standard,SAINTE-MARGUERITE
2,0803DPS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0803DPS\0803DPS - V2\CEINTURE_0803DPS_V2\No Team Assigned\2022_03_30-09_33_44\2022_03_30-09_33_44_BB_RR.csv,Standard,SAINTE-MARGUERITE
3,0804GOR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0804GOR\0804GOR - V2\CEINTURE_0804GOR_V2\No Team Assigned\2022_04_13-09_22_46\2022_04_13-09_22_46_BB_RR.csv,Réduction de la menace,SAINTE-MARGUERITE
4,0805BMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0805BMS\0805BMS - V2\CEINTURE_0805BMS_V2\2022_04_15-10_15_16_BB_RR.csv,Standard,SAINTE-MARGUERITE
5,0806KHS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0806KHS\V2\0806KHS - V2\CEINTURE_0806KHS_V2\No Team Assigned\2022_06_02-09_36_37\2022_06_02-09_36_37_BB_RR.csv,Standard,SAINTE-MARGUERITE
6,0807OMR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0807OMR\V2\0807OMR - V2\CEINTURE_0807OMR_V2\No Team Assigned\2022_06_24-09_39_18\2022_06_24-09_39_18_BB_RR.csv,Réduction de la menace,SAINTE-MARGUERITE
7,0808PJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0808PJR\0808PJR-V2\ceinture\No Team Assigned\2022_07_15-10_23_12\2022_07_15-10_23_12_BB_RR.csv,Réduction de la menace,SAINTE-MARGUERITE


## CGD

In [23]:
# ===== Racine CGD =====
racine_cgd = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD"
)

# ===== Candidats attendus =====
candidats_cgd = [
    c.upper() for c in resultats_globaux["CGD"]["valides"]
]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux["CGD"]["df"][
    ["Numero_inclusion", "Condition"]
].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()

dict_condition = dict(
    zip(df_conditions["Numero_inclusion"], df_conditions["Condition"])
)

rows_cgd = []

for root, dirs, files in os.walk(racine_cgd):

    if "V2" not in root.upper():
        continue

    for f in files:

        # CSV BB/BR_RR uniquement
        if not f.lower().endswith(".csv"):
            continue
        if not re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== identification du candidat =====
        pid = None
        for c in candidats_cgd:
            if c in chemin_upper:
                pid = c
                break

        if not pid:
            continue

        rows_cgd.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Condition": dict_condition.get(pid),
            "Feuille": "CGD"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_cgd = pd.DataFrame(rows_cgd)

df_cgd = (
    df_cgd
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus CGD : {len(candidats_cgd)}")
print(f"Fichiers retenus CGD   : {len(df_cgd)}")

manquants = sorted(
    set(candidats_cgd) - set(df_cgd["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats CGD sont présents")

display(df_cgd)


OK [0902SIS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0902SIS\0902SIS_V2\CEINTURE 0102SIS_V2\No Team Assigned\2022_05_19-09_43_14\2022_05_19-09_43_14_BB_RR.csv
OK [0904KJS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0904KJS\0904KJS - V2\CEINTURE_0904KJS_V2\No Team Assigned\2022_06_27-10_13_51\2022_06_27-10_13_51_BB_RR.csv
OK [0905SLR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0905SLR\0905SLR-V2\ceinture\No Team Assigned\2022_07_04-10_05_54\2022_07_04-10_05_54_BB_RR.csv
OK [0906DNR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0906DNR\0906DNR_V2\ceinture\No Team Assigned\2022_07_06-10_13_33\2022_07_06-10_13_33_BB_RR.csv

Candidats attendus CGD : 5
Fichiers retenus CGD   : 4
⚠️ Candidats manquants : ['0901SMR']


,Numero_inclusion,Chemin,Condition,Feuille
0,0902SIS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0902SIS\0902SIS_V2\CEINTURE 0102SIS_V2\No Team Assigned\2022_05_19-09_43_14\2022_05_19-09_43_14_BB_RR.csv,Standard,CGD
1,0904KJS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0904KJS\0904KJS - V2\CEINTURE_0904KJS_V2\No Team Assigned\2022_06_27-10_13_51\2022_06_27-10_13_51_BB_RR.csv,Standard,CGD
2,0905SLR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0905SLR\0905SLR-V2\ceinture\No Team Assigned\2022_07_04-10_05_54\2022_07_04-10_05_54_BB_RR.csv,Réduction de la menace,CGD
3,0906DNR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0906DNR\0906DNR_V2\ceinture\No Team Assigned\2022_07_06-10_13_33\2022_07_06-10_13_33_BB_RR.csv,Réduction de la menace,CGD


## CERCA

In [25]:
# ===== Racine CERCA =====
racine_cerca = r"C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS"

# ===== Candidats CERCA =====
candidats_cerca = [c.upper() for c in resultats_globaux['CERCA']['valides']]

# ===== Dictionnaire PID → Condition =====
df_conditions = resultats_globaux['CERCA']['df'][["Numero_inclusion", "Condition"]].copy()
df_conditions["Numero_inclusion"] = df_conditions["Numero_inclusion"].str.upper()
dict_condition = dict(zip(df_conditions["Numero_inclusion"], df_conditions["Condition"]))

# ===== Pattern candidats =====
patterns_candidats = {num: re.compile(rf"\b{num}\b", re.IGNORECASE) for num in candidats_cerca}

# ===== Collecte brute =====
fichiers_cerca = []

for root, dirs, files in os.walk(racine_cerca):
    if "V2" not in root.upper():
        continue

    for f in files:
        chemin = os.path.join(root, f)

        # Cas 1 : CSV classique
        if f.lower().endswith(".csv") and re.search(r"(BB_RR|BR_RR)", f, re.IGNORECASE):
            for num, pat in patterns_candidats.items():
                if pat.search(chemin):
                    fichiers_cerca.append(chemin)
                    break

        # Cas 2 : CSV dans zip
        elif f.lower().endswith(".zip"):
            try:
                with zipfile.ZipFile(chemin, 'r') as zf:
                    for name in zf.namelist():
                        if not name.lower().endswith(".csv"):
                            continue
                        if not re.search(r"(BB_RR|BR_RR)", name, re.IGNORECASE):
                            continue
                        full_path = f"{chemin}|{name}"  # garder le zip + fichier interne
                        for num, pat in patterns_candidats.items():
                            if pat.search(full_path):
                                fichiers_cerca.append(full_path)
                                break
            except zipfile.BadZipFile:
                print(f"⚠️ Zip corrompu : {chemin}")

# ===== DataFrame =====
rows = []

for num in candidats_cerca:
    fichiers_candidat = [f for f in fichiers_cerca if num in f]
    if fichiers_candidat:
        # Priorité aux fichiers contenant "ceinture"
        ceinture = [f for f in fichiers_candidat if "ceinture" in f.lower()]
        chemin_final = ceinture[0] if ceinture else fichiers_candidat[0]

        rows.append({
            "Numero_inclusion": num,
            "Chemin": chemin_final,
            "Condition": dict_condition.get(num),
            "Feuille": "CERCA"
        })

df_cerca = pd.DataFrame(rows)

# ===== Ajout manuel du candidat 0417KMS =====
chemin_0417KMS = r"C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\LEJEUNE\LEJEUNE\0417KMS\Ceinture\2022_11_24__09_03_09_BR_RR.csv"
if "0417KMS" in candidats_cerca and "0417KMS" not in df_cerca["Numero_inclusion"].values:
    df_cerca = pd.concat([
        df_cerca,
        pd.DataFrame([{
            "Numero_inclusion": "0417KMS",
            "Chemin": chemin_0417KMS,
            "Condition": dict_condition.get("0417KMS"),
            "Feuille": "CERCA"
        }])
    ], ignore_index=True)
    print("✅ Ajout manuel du candidat 0417KMS")

# ===== Résumé & contrôle =====
print(f"Candidats attendus CERCA : {len(candidats_cerca)}")
print(f"Fichiers BB/BR_RR retenus CERCA : {len(df_cerca)}")

manquants = sorted(set(candidats_cerca) - set(df_cerca["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats CERCA sont présents")

display(df_cerca)


✅ Ajout manuel du candidat 0417KMS
Candidats attendus CERCA : 11
Fichiers BB/BR_RR retenus CERCA : 8
⚠️ Candidats manquants : ['0420LMR', '0423RMR', '0424BLR']


,Numero_inclusion,Chemin,Condition,Feuille
0,0413MMR,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0413MMR\V2\Ceinture\2019_12_18__08_20_29_BR_RR.csv,Réduction de la menace,CERCA
1,0414PVR,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0414PVR\V2\Ceinture\2020_01_07__09_15_42_BR_RR.csv.zip|2020_01_07__09_15_42_BR_RR.csv,Réduction de la menace,CERCA
2,0415HJS,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0415HJS\V2\Ceinture\2020_02_11__09_43_13_BR_RR.csv,Standard,CERCA
3,0417FAS,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0417FAS\V2\Ceinture\2020_02_12__10_41_16_BR_RR.csv,Standard,CERCA
4,0418MPR,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0418MPR\V2\Ceinture\2020_02_13__09_33_23_BR_RR.csv,Réduction de la menace,CERCA
5,0420MCR,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0420MCR\V2\Ceinture\2020_02_28__09_44_26_BR_RR.csv,Réduction de la menace,CERCA
6,0421ACS,C:\Users\judupont\Desktop\AGING_19_01_2026\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0421ACS\V2\Ceinture\2020_03_04__10_12_00_BR_RR.csv,Standard,CERCA
7,0417KMS,C:\Users\judupont\Desktop\AGING_19_01_2026\CHU-POITIERS\LEJEUNE\LEJEUNE\0417KMS\Ceinture\2022_11_24__09_03_09_BR_RR.csv,Standard,CERCA


## CONCATENATION

In [26]:
# ===== Liste des DataFrames par feuille =====
dfs = [
    df_aphm,  
    df_caen_final,
    df_poitiers_final,     
    df_rouen_final,
    df_laveran_final,
    df_lpc_final,         
    df_lpc2,
    df_sm,          
    df_cgd,         
    df_cerca      
]

# ===== Normalisation des colonnes =====
for i, df in enumerate(dfs):
    # Certains DF ont "PID" au lieu de "Numero_inclusion"
    if "PID" in df.columns:
        df.rename(columns={"PID": "Numero_inclusion"}, inplace=True)
    # Assurer que la colonne Condition existe
    if "Condition" not in df.columns:
        df["Condition"] = None
    # On garde uniquement les colonnes essentielles
    df = df[["Numero_inclusion", "Chemin", "Feuille", "Condition"]]
    dfs[i] = df

# ===== Concatenation =====
df_global = pd.concat(dfs, ignore_index=True)

# ===== Suppression des doublons par Numero_inclusion =====
df_global = df_global.drop_duplicates(subset=["Numero_inclusion"], keep="first")

# ===== Tri par Numero_inclusion =====
df_global = df_global.sort_values("Numero_inclusion").reset_index(drop=True)

# ===== Résumé =====
print(f"Total candidats uniques : {df_global['Numero_inclusion'].nunique()}")
print(f"Total fichiers conservés : {len(df_global)}")
display(df_global)


Total candidats uniques : 156
Total fichiers conservés : 156


,Numero_inclusion,Chemin,Feuille,Condition
0,0101CAR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0101CAR\0101CAR 06-07-2018 V2\Ceinture 0101CAR\2018_07_06-08_24_06_BB_RR.csv,APHM,Réduction de la menace
1,0101EMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\V2\Ceinture\2019_01_22-10_10_43\2019_01_22-10_10_43_BB_RR.csv,LPC,Standard
2,0102PCR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0102PCR\0102PCR 09-07-2018 V2\Ceinture 0102PCR\2018_07_09-08_57_05_BB_RR.csv,APHM,Réduction de la menace
3,0103BPS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V2\Ceinture\2018_12_11-10_08_18\2018_12_11-10_08_18_BB_RR.csv,LPC,Standard
4,0103SHS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0103SHS\0103SHS 10-09-2018 V2\Ceinture 0103SHS\2018_09_10-09_03_59_BB_RR.csv,APHM,Standard
5,0104FJS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0104FJS\0104FJS 29-11-2018 V2\Ceinture 0104FJS\2018_11_29-08_27_38_BB_RR.csv,APHM,Standard
6,0104IBS,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0104IBS\V2\Ceinture\2018_11_28-09_57_59\2018_11_28-09_57_59_BB_RR.csv,LPC,Standard
7,0105PNR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0105PNR\0105PNR 10-12-2018 V2\Ceinture 0105PNR\2018_12_10-08_23_34_BB_RR.csv,APHM,Réduction de la menace
8,0106DJR,C:\Users\judupont\Desktop\AGING_19_01_2026\LNSC\DESHAYES\DESHAYES\DATA_Aging\0106DJR\V2\Ceinture\2018_12_19-10_05_30\2018_12_19-10_05_30_BB_RR.csv,LPC,Réduction de la menace
9,0106JLS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0106JLS\0106JLS 03-01-2019 V2\Ceinture 0106JLS\2019_01_03-08_25_39_BB_RR.csv,APHM,Standard


In [27]:
# CONSERVATION DES FEUILLES ET CONDITIONS DANS UN FICHIER TXT 

chemin_txt = r"C:\Users\judupont\Desktop\df_global.txt"

df_global.to_csv(
    chemin_txt,
    sep="\t",
    index=False,
    encoding="utf-8-sig"
)

print(f"📄 df_global sauvegardé : {chemin_txt}")

📄 df_global sauvegardé : C:\Users\judupont\Desktop\df_global.txt


# Verification que le temps d'experience et le temps du fichier BB_RR correspondent : offset 

In [28]:
def convertir_heure_en_secondes(x):
    if pd.isna(x):
        return None

    if isinstance(x, (int, float)):
        seconds = float(x) * 24 * 3600
        return seconds % (24 * 3600)

    try:
        t = pd.to_datetime(str(x).strip()).time()
        return t.hour * 3600 + t.minute * 60 + t.second
    except Exception:
        return None


# ===== NORMALISATION GLOBALE =====
df_global["Numero_inclusion"] = (
    df_global["Numero_inclusion"]
    .astype(str)
    .str.strip()
    .str.upper()
)

set_global = set(df_global["Numero_inclusion"])


dfs_excel = []

for feuille, data in resultats_bloc1.items():

    df_excel = data["df"].copy()

    # ===== NORMALISATION DES NOMS =====
    df_excel["Numero_inclusion"] = (
        df_excel["Numero_inclusion"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    df_excel = df_excel[
        df_excel["Numero_inclusion"].isin(set_global)
    ]

    if df_excel.empty:
        continue

    # ===== Conversion heures =====
    df_excel["heure_debut_sec"] = df_excel["heure_ceinture_v2"].apply(convertir_heure_en_secondes)
    df_excel["heure_fin_sec"] = df_excel["heure_fin_tests_v2"].apply(convertir_heure_en_secondes)

    # ===== Calcul de la durée de la visite =====
    df_excel["duree_experience_sec"] = (
        df_excel["heure_fin_sec"] - df_excel["heure_debut_sec"]
    )

    df_excel.loc[
        df_excel["duree_experience_sec"] < 0,
        "duree_experience_sec"
    ] += 24 * 3600

    df_excel["Feuille"] = feuille
    dfs_excel.append(df_excel)


if dfs_excel:
    df_excel_global = pd.concat(dfs_excel, ignore_index=True)
else:
    df_excel_global = pd.DataFrame()
    
print(f"Lignes Excel retenues : {len(df_excel_global)}")
display(df_excel_global[[
    "Numero_inclusion",
    "Feuille",
    "duree_experience_sec"
]])

Lignes Excel retenues : 156


,Numero_inclusion,Feuille,duree_experience_sec
0,0101CAR,APHM,9480
1,0102PCR,APHM,7620
2,0103SHS,APHM,5580
3,0105PNR,APHM,7260
4,0104FJS,APHM,9720
5,0106JLS,APHM,7560
6,0107DSS,APHM,7260
7,0108BFS,APHM,7920
8,0109GSS,APHM,7560
9,0110LPR,APHM,8160


In [29]:
# ==== Conversion Date dans Excel ====
df_excel_global["Date_v2_dt"] = pd.to_datetime(
    df_excel_global["Date_v2"], errors="coerce"
).dt.date

# ==== Candidats à ignorer pour la date (verifier à la main) ====
ignore_date_check = [
   "0403DCR", "0502LIR", "0507BDS", "0124BAR", "0125BMR", "0131CPS", "0414PVR"
]

rows_rr = []

for _, row in df_global.iterrows():

    pid = str(row["Numero_inclusion"]).strip().upper()
    chemin = row["Chemin"]

    # ---- récupérer ligne Excel ----
    ligne_excel = df_excel_global[df_excel_global["Numero_inclusion"] == pid]
    if ligne_excel.empty:
        continue

    date_v2 = ligne_excel.iloc[0]["Date_v2_dt"]
    duree_excel_sec = ligne_excel.iloc[0]["duree_experience_sec"]

    # ---- lecture CSV BB_RR (ZIP ou non) ----
    try:
        if ".ZIP|" in chemin.upper():
            zip_path, inner_csv = chemin.split("|", 1)

            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(inner_csv) as f:
                    df_rr = pd.read_csv(f)

        else:
            df_rr = pd.read_csv(chemin)

    except Exception as e:
        print(f"⚠️ Lecture impossible RR pour {pid} : {e}")
        continue

    if "Timestamp" not in df_rr.columns:
        continue

    # ---- conversion timestamp CSV ----
    df_rr["Timestamp_dt"] = pd.to_datetime(
        df_rr["Timestamp"].astype(str).str.split(".").str[0],
        dayfirst=True,
        errors="coerce"
    )

    df_rr = df_rr.dropna(subset=["Timestamp_dt"])

    # ---- filtrage date ----
    if pid not in ignore_date_check and pd.notna(date_v2):
        df_rr = df_rr[df_rr["Timestamp_dt"].dt.date == date_v2]

    if df_rr.empty:
        continue

    # ---- durée RR ----
    t_min = df_rr["Timestamp_dt"].dt.floor("s").min()
    t_max = df_rr["Timestamp_dt"].dt.floor("s").max()
    duree_rr_sec = (t_max - t_min).total_seconds()

    rows_rr.append({
        "Numero_inclusion": pid,
        "duree_rr_sec": duree_rr_sec,
        "duree_excel_sec": duree_excel_sec,
        "nb_points": len(df_rr),
    })

df_rr_global = pd.DataFrame(rows_rr)

print(f"RR calculés : {len(df_rr_global)}")

display(df_rr_global)

RR calculés : 154


,Numero_inclusion,duree_rr_sec,duree_excel_sec,nb_points
0,0101CAR,9577.0,9480,171017
1,0101EMS,6311.0,6240,112696
2,0102PCR,7770.0,7620,138750
3,0103BPS,5227.0,5160,93339
4,0103SHS,5672.0,5580,101285
5,0104FJS,9913.0,9720,177017
6,0104IBS,5740.0,5640,102500
7,0105PNR,7475.0,7260,133482
8,0106DJR,6666.0,6600,119035
9,0106JLS,7691.0,7560,137339


In [30]:
excel_ids = set(df_excel_global["Numero_inclusion"])
rr_ids = set(df_rr_global["Numero_inclusion"])

manquants_dans_rr = excel_ids - rr_ids
manquants_dans_excel = rr_ids - excel_ids

print("Présents dans Excel mais pas dans RR :", manquants_dans_rr)
print("Présents dans RR mais pas dans Excel :", manquants_dans_excel)
print("Manquants dans RR :", sorted(excel_ids - rr_ids))
print("Manquants dans Excel :", sorted(rr_ids - excel_ids))

Présents dans Excel mais pas dans RR : {'0323RFS', '0515FAR'}
Présents dans RR mais pas dans Excel : set()
Manquants dans RR : ['0323RFS', '0515FAR']
Manquants dans Excel : []


# D'ou vient l'écart ? L'offest doit etre appliqué en debut, en fin, au milieu?

In [31]:
def sec_to_datetime(sec, date_ref):
    return datetime.combine(
        date_ref.date(),
        datetime.min.time()
    ) + timedelta(seconds=int(sec))

resultats_offset = []

# Ces candidats ont leur temps de début d'enregistrement décaler de pile 1h mais la durée de leur visite est la meme que celle de l'acquisiton
# donc on peut décaler artificiellement leur heure de début pour qu'elle soit identique a celle indiquée sur le Excel
candidats_rebase_excel = {
    "0136CJR",    
    "0341LJS",
    "0512RCR"
}

for _, row in df_global.iterrows():

    pid = str(row["Numero_inclusion"]).strip().upper()
    chemin_bb_rr = row["Chemin"]

    # ===== Lecture BB_RR (ZIP ou non) =====
    try:
        if ".ZIP|" in chemin_bb_rr.upper():
            zip_path, inner_csv = chemin_bb_rr.split("|", 1)
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(inner_csv) as f:
                    df_rr = pd.read_csv(f)
        else:
            df_rr = pd.read_csv(chemin_bb_rr)
    except Exception as e:
        print(f"⚠️ Lecture BB_RR impossible pour {pid} : {e}")
        continue

    if "Timestamp" not in df_rr.columns:
        continue

    # ===== Parsing timestamp =====
    df_rr["Timestamp"] = pd.to_datetime(
        df_rr["Timestamp"].astype(str).str.split(".").str[0],
        dayfirst=True,
        errors="coerce"
    )
    df_rr = df_rr.dropna(subset=["Timestamp"])
    if df_rr.empty:
        continue

    t_min = df_rr["Timestamp"].min()
    t_max = df_rr["Timestamp"].max()

    # ===== Excel =====
    ligne_excel = df_excel_global.loc[
        df_excel_global["Numero_inclusion"] == pid
    ]
    if ligne_excel.empty:
        continue

    duree_excel_sec = ligne_excel["duree_experience_sec"].values[0]
    if pd.isna(duree_excel_sec):
        continue

    # ===== Cas spécial : rebase Excel sur RR =====
    if pid in candidats_rebase_excel:
        h_debut = t_min
        h_fin = h_debut + pd.to_timedelta(duree_excel_sec, unit="s")
        mode = "REBASE_RR"
        # 🔹 delta_retard_sec pour info, même si rebase
        delta_retard_sec = 0
    else:
        h_debut_sec = ligne_excel["heure_debut_sec"].values[0]
        h_fin_sec = ligne_excel["heure_fin_sec"].values[0]
        if pd.isna(h_debut_sec) or pd.isna(h_fin_sec):
            continue

        h_debut = sec_to_datetime(h_debut_sec, t_min)
        h_fin   = sec_to_datetime(h_fin_sec, t_min)
        mode = "STANDARD"

        # 🔥 Calcul du delta retard sec comme pour les EDA
        delta_retard_sec = max((t_min - h_debut).total_seconds(), 0)

    # ===== Comptages =====
    nb_avant = (df_rr["Timestamp"] < h_debut).sum()
    nb_dans = ((df_rr["Timestamp"] >= h_debut) & (df_rr["Timestamp"] <= h_fin)).sum()
    nb_apres = (df_rr["Timestamp"] > h_fin).sum()
    total = len(df_rr)

    # ===== Diagnostic =====
    if nb_apres > nb_avant:
        origine = "FIN"
    elif nb_avant > nb_apres:
        origine = "DEBUT"
    else:
        origine = "MIXTE"

    resultats_offset.append({
        "Numero_inclusion": pid,
        "nb_total_points": total,
        "nb_avant_excel": nb_avant,
        "nb_dans_excel": nb_dans,
        "nb_apres_excel": nb_apres,
        "delta_retard_sec": delta_retard_sec  
    })

df_offset_bb_rr = pd.DataFrame(resultats_offset)

print(f"Offsets BB_RR calculés : {len(df_offset_bb_rr)}")
display(df_offset_bb_rr)

Offsets BB_RR calculés : 156


,Numero_inclusion,nb_total_points,nb_avant_excel,nb_dans_excel,nb_apres_excel,delta_retard_sec
0,0101CAR,171017,0,169189,1828,6.0
1,0101EMS,112696,0,110671,2025,43.0
2,0102PCR,138750,0,135993,2757,5.0
3,0103BPS,93339,0,91832,1507,18.0
4,0103SHS,101285,0,98600,2685,59.0
5,0104FJS,177017,0,172904,4113,38.0
6,0104IBS,102500,11,100732,1757,0.0
7,0105PNR,133482,0,129046,4436,34.0
8,0106DJR,119035,0,116261,2774,90.0
9,0106JLS,137339,0,134314,3025,39.0


In [32]:
# ===== Dossier de sortie =====
desktop = Path.home() / "Desktop"
output_dir = desktop / "bb_rr_tronqués_v2"
output_dir.mkdir(exist_ok=True)

print(f"📁 Dossier de sortie : {output_dir}")

# ===== Boucle troncature =====
for _, row in df_offset_bb_rr.iterrows():

    pid = str(row["Numero_inclusion"]).strip().upper()

    # récupérer chemin BB_RR depuis df_global
    ligne_global = df_global[df_global["Numero_inclusion"] == pid]
    if ligne_global.empty:
        continue

    chemin_bb_rr = ligne_global.iloc[0]["Chemin"]

    nb_avant = int(row["nb_avant_excel"])
    nb_apres = int(row["nb_apres_excel"])

    # ===== Lecture BB_RR (ZIP ou non) =====
    try:
        if ".ZIP|" in chemin_bb_rr.upper():
            zip_path, inner_csv = chemin_bb_rr.split("|", 1)

            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(inner_csv) as f:
                    df_rr = pd.read_csv(f)
        else:
            df_rr = pd.read_csv(chemin_bb_rr)

    except Exception as e:
        print(f"❌ Lecture impossible {pid}: {e}")
        continue

    total = len(df_rr)

    # ===== Indices de coupe =====
    start = nb_avant
    end = total - nb_apres

    if start >= end:
        print(f"⚠️ Troncature invalide pour {pid} (start={start}, end={end})")
        continue

    df_rr_trunc = df_rr.iloc[start:end].reset_index(drop=True)

    if df_rr_trunc.empty:
        print(f"⚠️ {pid} → fichier vide après troncature")
        continue

    # ===== Nom fichier de sortie =====
    if ".ZIP|" in chemin_bb_rr.upper():
        nom_base = Path(inner_csv).name
    else:
        nom_base = Path(chemin_bb_rr).name

    nom_fichier = f"offset_{pid}_{nom_base}"
    path_sortie = output_dir / nom_fichier

    # ===== Sauvegarde =====
    df_rr_trunc.to_csv(path_sortie, index=False)

    print(
        f"✅ {pid} | "
        f"avant={nb_avant}, après={nb_apres} | "
        f"{len(df_rr_trunc)} lignes → {path_sortie.name}"
    )

print("🎯 Troncature terminée")

📁 Dossier de sortie : C:\Users\judupont\Desktop\bb_rr_tronqués_v2
✅ 0101CAR | avant=0, après=1828 | 169189 lignes → offset_0101CAR_2018_07_06-08_24_06_BB_RR.csv
✅ 0101EMS | avant=0, après=2025 | 110671 lignes → offset_0101EMS_2019_01_22-10_10_43_BB_RR.csv
✅ 0102PCR | avant=0, après=2757 | 135993 lignes → offset_0102PCR_2018_07_09-08_57_05_BB_RR.csv
✅ 0103BPS | avant=0, après=1507 | 91832 lignes → offset_0103BPS_2018_12_11-10_08_18_BB_RR.csv
✅ 0103SHS | avant=0, après=2685 | 98600 lignes → offset_0103SHS_2018_09_10-09_03_59_BB_RR.csv
✅ 0104FJS | avant=0, après=4113 | 172904 lignes → offset_0104FJS_2018_11_29-08_27_38_BB_RR.csv
✅ 0104IBS | avant=11, après=1757 | 100732 lignes → offset_0104IBS_2018_11_28-09_57_59_BB_RR.csv
✅ 0105PNR | avant=0, après=4436 | 129046 lignes → offset_0105PNR_2018_12_10-08_23_34_BB_RR.csv
✅ 0106DJR | avant=0, après=2774 | 116261 lignes → offset_0106DJR_2018_12_19-10_05_30_BB_RR.csv
✅ 0106JLS | avant=0, après=3025 | 134314 lignes → offset_0106JLS_2019_01_03-08_2

In [33]:
# ===== Paramètres =====
min_lignes = 70000
candidats_a_exclure = ["0323FRS", "0515FAR"]  # liste de candidats à exclure 0323 c'est la v4 et 0515 valerus abberantes 

dossier = Path.home() / "Desktop" / "bb_rr_tronqués_v2"
print(f"📂 Dossier analysé : {dossier}")

supprimes = []
conserves = []

for fichier in dossier.glob("*.csv"):
    nom = fichier.name

    # ===== Exclusion candidat spécifique =====
    if any(candidat in nom for candidat in candidats_a_exclure):
        fichier.unlink()
        supprimes.append((nom, "candidat exclu"))
        print(f"🗑️ {nom} → candidat exclu")
        continue

    # ===== Lecture rapide =====
    try:
        n_lignes = sum(1 for _ in open(fichier, "r", encoding="utf-8")) - 1
    except Exception as e:
        print(f"❌ Lecture impossible {nom}: {e}")
        continue

    # ===== Exclusion fichiers trop courts =====
    if n_lignes < min_lignes:
        fichier.unlink()
        supprimes.append((nom, f"{n_lignes} lignes"))
        print(f"❌ {nom} → trop petit ({n_lignes} lignes)")
    else:
        conserves.append((nom, n_lignes))
        print(f"✅ {nom} → conservé ({n_lignes} lignes)")

print("\n====== RÉSUMÉ ======")
print(f"Fichiers supprimés : {len(supprimes)}")
print(f"Fichiers conservés : {len(conserves)}")

📂 Dossier analysé : C:\Users\judupont\Desktop\bb_rr_tronqués_v2
✅ offset_0101CAR_2018_07_06-08_24_06_BB_RR.csv → conservé (169189 lignes)
✅ offset_0101EMS_2019_01_22-10_10_43_BB_RR.csv → conservé (110671 lignes)
✅ offset_0102PCR_2018_07_09-08_57_05_BB_RR.csv → conservé (135993 lignes)
✅ offset_0103BPS_2018_12_11-10_08_18_BB_RR.csv → conservé (91832 lignes)
✅ offset_0103SHS_2018_09_10-09_03_59_BB_RR.csv → conservé (98600 lignes)
✅ offset_0104FJS_2018_11_29-08_27_38_BB_RR.csv → conservé (172904 lignes)
✅ offset_0104IBS_2018_11_28-09_57_59_BB_RR.csv → conservé (100732 lignes)
✅ offset_0105PNR_2018_12_10-08_23_34_BB_RR.csv → conservé (129046 lignes)
✅ offset_0106DJR_2018_12_19-10_05_30_BB_RR.csv → conservé (116261 lignes)
✅ offset_0106JLS_2019_01_03-08_25_39_BB_RR.csv → conservé (134314 lignes)
✅ offset_0107DSS_2019_01_14-08_24_42_BB_RR.csv → conservé (129661 lignes)
✅ offset_0108BFS_2019_01_17-08_42_57_BB_RR.csv → conservé (141446 lignes)
✅ offset_0108MMS_2019_01_24-09_57_32_BB_RR.csv → c

# On peut passer au découpage!

In [35]:
'''
Nous pouvons désormais appliquer le découpage présenté au début de ce rapport (figures 2 et 3).
La procédure est la suivante :
— Bloc 1 : commence à la première ligne du fichier tronqué, pour les deux types de signaux. Nous
partons du début et ajoutons la durée du bloc 1, calculée en amont.
— Bloc 2 : ne débute pas immédiatement après la fin du bloc 1. Pour la visite 2, il commence
après le visionnage de la vidéo, et pour la visite 4, après l’exercice d’écriture. Nous décalons
donc le début du bloc 2 de ces durées, puis nous ajoutons la durée du bloc 2.
— Bloc 3 : commence immédiatement à la fin du bloc 2 pour tous les fichiers. Nous ajoutons
simplement la durée du bloc 3.
'''

def decouper_bb_rr_par_bloc(
    df_rr,
    duree_bloc1,
    duree_video,
    duree_bloc2,
    duree_bloc3,
    delta
):
    df_rr = df_rr.copy()

    # ===== Timestamp propre =====
    df_rr["Timestamp"] = pd.to_datetime(
        df_rr["Timestamp"], dayfirst=True, errors="coerce"
    )
    df_rr = df_rr.dropna(subset=["Timestamp"])

    if df_rr.empty:
        return None

    # ===== Temps de référence =====
    t0 = df_rr["Timestamp"].min()
    t_fin_rr = df_rr["Timestamp"].max()

    # ===== Bornes temporelles =====
    fin_bloc1 = t0 + pd.Timedelta(seconds=max(0, duree_bloc1 - delta))
    debut_bloc2 = fin_bloc1 + pd.Timedelta(seconds=duree_video)
    fin_bloc2 = debut_bloc2 + pd.Timedelta(seconds=duree_bloc2)
    fin_bloc3 = fin_bloc2 + pd.Timedelta(seconds=duree_bloc3)

    # Sécurité (ne jamais dépasser le fichier)
    fin_bloc3 = min(fin_bloc3, t_fin_rr)

    

    return {
        "bloc1": df_rr[
            (df_rr["Timestamp"] >= t0) &
            (df_rr["Timestamp"] < fin_bloc1)
        ],
        "bloc2": df_rr[
            (df_rr["Timestamp"] >= debut_bloc2) &
            (df_rr["Timestamp"] < fin_bloc2)
        ],
        "bloc3": df_rr[
            (df_rr["Timestamp"] >= fin_bloc2) &
            (df_rr["Timestamp"] < fin_bloc3)
        ],
        "bornes": {
            "t0": t0,
            "fin_bloc1": fin_bloc1,
            "debut_bloc2": debut_bloc2,
            "fin_bloc2": fin_bloc2,
            "fin_bloc3": fin_bloc3,
            "t_fin_rr": t_fin_rr
        }
    }

In [36]:
def convertir_et_normaliser_heure(x):
    """
    Convertit n'importe quel format d'heure Excel / texte / datetime en pd.Timestamp
    normalisé sur le 1900-01-01, ne gardant que l'heure, minute, seconde.
    """
    if pd.isna(x):
        return pd.NaT

    # Excel numérique → fraction de jour ou date complète
    if isinstance(x, (int, float)):
        seconds = (float(x) * 24 * 3600) % (24*3600)
        return pd.Timestamp("1900-01-01") + pd.to_timedelta(seconds, unit="s")

    # Timestamp ou datetime → on garde juste l'heure
    if isinstance(x, (pd.Timestamp, datetime.datetime)):
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=x.hour, minute=x.minute, second=x.second
        )

    # Texte
    try:
        t = pd.to_datetime(str(x).strip(), errors="coerce")
        if pd.isna(t):
            return pd.NaT
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=t.hour, minute=t.minute, second=t.second
        )
    except Exception:
        return pd.NaT


In [37]:
def get_durees_blocs(pid, feuille):
    """
    Récupère les durées pour les 3 blocs et la vidéo pour un Numero_inclusion donné et une feuille donnée.
    Applique un correctif spécifique pour 0101EMS car il ya un problème dans la conversion en temps d'une de ses cases
    """
    pid = str(pid).upper()

    # ===== Bloc 1 =====
    df_b1 = pd.DataFrame(resultats_bloc1[feuille]["df"])
    ligne = df_b1.loc[df_b1["Numero_inclusion"].str.upper() == pid]

    if ligne.empty:
        raise ValueError(f"PID {pid} absent du bloc 1 dans la feuille {feuille}")

    d1 = ligne["duree_bloc1"].values[0]
    condition = ligne["Condition"].values[0]

    # ===== Durée vidéo =====
    t_fin_anamnese = convertir_et_normaliser_heure(
        ligne["heure_fin_anamnese_v2"].values[0]
    )
    t_debut_rlri = convertir_et_normaliser_heure(
        ligne["heure_rlri16imm_debut_V2"].values[0]
    )

    if pd.notna(t_fin_anamnese) and pd.notna(t_debut_rlri):
        delta = (t_debut_rlri - t_fin_anamnese) / pd.Timedelta(seconds=1)  # en secondes
        if delta < 0:
            delta += 24*3600
    else:
        delta = 0
    duree_video_sec = delta

    # ===== Bloc 2 =====
    if condition == "Standard":
        df_b2 = pd.DataFrame(resultats_bloc2_S[feuille]["df"])
        duree_col = "duree_bloc2S"
    else:
        df_b2 = pd.DataFrame(resultats_bloc2_R[feuille]["df"])
        duree_col = "duree_bloc2R"

    # ===== Récupérer d2 normalement =====
    d2 = df_b2.loc[df_b2["Numero_inclusion"].str.upper() == pid, duree_col].values[0]

    # ===== Bloc 3 =====
    df_b3 = pd.DataFrame(resultats_bloc3[feuille]["df"])
    d3 = df_b3.loc[df_b3["Numero_inclusion"].str.upper() == pid, "duree_bloc3"].values[0]

    return {
        "bloc1_sec": d1 * 60,
        "video_sec": duree_video_sec,
        "bloc2_sec": d2 * 60,
        "bloc3_sec": d3 * 60,
        "condition": condition
    }

In [38]:
# ===== Dossiers =====
input_dir = os.path.join(os.path.expanduser("~"), "Desktop", "bb_rr_tronqués_v2")
output_dir = os.path.join(os.path.expanduser("~"), "Desktop", "bb_rr_tronque_blocs_v2")
os.makedirs(output_dir, exist_ok=True)

resultats_decoupage = {}

# ===== Boucle UNIQUEMENT sur les fichiers tronqués =====
for fichier in os.listdir(input_dir):

    if not fichier.lower().endswith(".csv"):
        continue

    chemin_bb_rr = os.path.join(input_dir, fichier)

    # ===== Extraction PID depuis le nom =====
    match = re.search(r"\d{4}[A-Z]{3}", fichier.upper())
    if not match:
        print(f"⚠️ PID introuvable dans {fichier}")
        continue

    pid = match.group()

    # ===== Récupération de la feuille associée =====
    ligne = df_global.loc[df_global["Numero_inclusion"] == pid]
    if ligne.empty:
        print(f"⚠️ Feuille introuvable pour {pid}")
        continue

    feuille = ligne["Feuille"].values[0]

    print(f"\n--- Découpage {pid} ({feuille}) ---")

    # ===== Lecture BB_RR =====
    try:
        df_rr = pd.read_csv(chemin_bb_rr)
    except Exception as e:
        print(f"❌ Lecture impossible {pid} : {e}")
        continue

    # ===== Récupération des durées =====
    try:
        durees = get_durees_blocs(pid, feuille)
    except Exception as e:
        print(f"⚠️ Impossible de récupérer les durées pour {pid} ({feuille}) : {e}")
        continue

    delta_row = df_offset_bb_rr.loc[df_offset_bb_rr["Numero_inclusion"].str.upper() == pid.upper()]
    if delta_row.empty:
        delta_val = 0
    else:
        delta_val = float(delta_row["delta_retard_sec"].values[0])
        
    # ===== Découpage =====
    decoupe = decouper_bb_rr_par_bloc(
        df_rr=df_rr,
        duree_bloc1=durees["bloc1_sec"],
        duree_video=durees["video_sec"],
        duree_bloc2=durees["bloc2_sec"],
        duree_bloc3=durees["bloc3_sec"],
        delta = delta_val
    )

    if decoupe is None:
        print(f"⚠️ Découpage vide pour {pid}")
        continue

    # ===== Sauvegarde par bloc =====
    for bloc in ["bloc1", "bloc2", "bloc3"]:
        df_bloc = decoupe[bloc]

        if df_bloc.empty:
            print(f"⚠️ {pid} {bloc} vide")
            continue

        nom_sortie = f"{pid}_{bloc}.csv"
        df_bloc.to_csv(
            os.path.join(output_dir, nom_sortie),
            index=False
        )

    # ===== Stockage mémoire =====
    resultats_decoupage[pid] = decoupe

    print(
        f"✔ {pid} | "
        f"B1={len(decoupe['bloc1'])} | "
        f"B2={len(decoupe['bloc2'])} | "
        f"B3={len(decoupe['bloc3'])}"
    )

print("\n✅ Découpage terminé")
print("📁 Résultats dans :", output_dir)


--- Découpage 0101CAR (APHM) ---
✔ 0101CAR | B1=27750 | B2=68572 | B3=64286

--- Découpage 0101EMS (LPC) ---
✔ 0101EMS | B1=23875 | B2=39643 | B3=45000

--- Découpage 0102PCR (APHM) ---
✔ 0102PCR | B1=28840 | B2=45000 | B3=53572

--- Découpage 0103BPS (LPC) ---
✔ 0103BPS | B1=12536 | B2=36428 | B3=41786

--- Découpage 0103SHS (APHM) ---
✔ 0103SHS | B1=10733 | B2=40714 | B3=46072

--- Découpage 0104FJS (APHM) ---
✔ 0104FJS | B1=30393 | B2=66429 | B3=73928

--- Découpage 0104IBS (LPC) ---
✔ 0104IBS | B1=17143 | B2=34286 | B3=45000

--- Découpage 0105PNR (APHM) ---
✔ 0105PNR | B1=26179 | B2=43929 | B3=50357

--- Découpage 0106DJR (LPC) ---
✔ 0106DJR | B1=17679 | B2=40714 | B3=50357

--- Découpage 0106JLS (APHM) ---
✔ 0106JLS | B1=17518 | B2=72857 | B3=42857

--- Découpage 0107DSS (APHM) ---
✔ 0107DSS | B1=20358 | B2=61071 | B3=47143

--- Découpage 0108BFS (APHM) ---
✔ 0108BFS | B1=17143 | B2=53571 | B3=69643

--- Découpage 0108MMS (LPC) ---
✔ 0108MMS | B1=10143 | B2=36428 | B3=48215

---

In [ ]:
## Traitment pour passer le fichier dans kubios